<div style="font-family: Arial, sans-serif; border-left: 10px solid #C2410C; background-color: #1F2937; padding: 15px 25px; border-radius: 4px; max-width: 100%; box-sizing: border-box;">
    <h1 style="color: #FBBF24; margin: 0; font-size: 24px; letter-spacing: 1px;">
        📊 Análisis de Tickets de Soporte al Cliente
    </h1>
    <p style="color: #FEF3C7; margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;">
        Proyecto Final | NLP & Deep Learning | Comisión 77805 | <b>Morales, Luis Emanuel</b>
    </p>
</div>

<div style="font-family: Arial, sans-serif; padding: 10px 20px; border-left: 10px solid #FBBF24; background-color: #FEF3C7; border-radius: 4px; max-width: 100%; box-sizing: border-box;">
    <h3 style="color: #C2410C; margin: 0 0 8px 0; font-size: 18px;">📍 Introducción y problemática</h3>
    <p style="color: #1F2937; font-size: 14px; line-height: 1.5; margin: 0;">
        Este proyecto tiene como objetivo construir un pipeline de clasificación de texto para categorizar automáticamente tickets de soporte. La elección del <b>Customer Support Tickets Dataset</b> se fundamenta en su diversidad lingüística y en la necesidad de procesar descripciones narrativas complejas, superando los 2.000 registros requeridos para garantizar un entrenamiento robusto de modelos de Deep Learning.
        <br><br>
        <b>Desafío Técnico:</b> Transformar texto desestructurado en tensores numéricos mediante un flujo que integra limpieza por expresiones regulares, normalización con NLTK/spaCy y modelado con redes neuronales densas en PyTorch.
    </p>
</div>

In [ ]:
# ==============================================================================
# INSTALACIÓN DE LIBRERÍAS
# ==============================================================================
# !pip install pandas numpy matplotlib seaborn
# !pip install nltk spacy scikit-learn
# !pip install torch torchvision torchaudio
# !python -m spacy download es_core_news_sm  # Para procesar texto en español
# !python -m spacy download en_core_web_sm   # Para procesar texto en inglés

# Descomentar y procesar lo que sea necesario

In [ ]:
# Librerías básicas y manejo de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Procesamiento de Lenguaje Natural (NLP)
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Machine Learning y Modelado
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning con PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Configuración de NLTK (Descarga de recursos necesarios)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Configuración de visualización
%matplotlib inline
sns.set_theme(style="whitegrid")

In [15]:
# Carga del dataset
df = pd.read_csv('customer_support_tickets.csv')

# Verificación de registros
print(f"Total de registros: {len(df)}")
print(f"Dimensiones del dataset: {df.shape}")
print(f"Columnas disponibles: {df.columns}")


Total de registros: 200000
Dimensiones del dataset: (200000, 30)
Columnas disponibles: Index(['ticket_id', 'customer_name', 'customer_email', 'product', 'category',
       'issue_description', 'resolution_notes', 'priority', 'status',
       'channel', 'region', 'customer_age', 'customer_gender',
       'subscription_type', 'customer_tenure_months', 'previous_tickets',
       'customer_satisfaction_score', 'first_response_time_hours',
       'resolution_time_hours', 'ticket_created_date', 'ticket_resolved_date',
       'escalated', 'sla_breached', 'operating_system', 'browser',
       'payment_method', 'language', 'preferred_contact_time',
       'issue_complexity_score', 'customer_segment'],
      dtype='object')


In [25]:
print(df['category'].value_counts(normalize=True) * 100)

category
Feature Request              10.0845
Subscription Cancellation    10.0480
Performance Issue            10.0370
Security Concern             10.0200
Login Issue                  10.0010
Payment Problem               9.9985
Bug Report                    9.9905
Refund Request                9.9500
Data Sync Issue               9.9385
Account Suspension            9.9320
Name: proportion, dtype: float64


In [19]:
# Selección de las columnas para el NLP
# En este dataset, 'Ticket Description' es nuestro texto y 'Ticket Type'
df_nlp = df[['issue_description', 'category']].copy()

# Eliminar nulos si los hubiera
df_nlp.dropna(inplace=True)

df_nlp.head()

,issue_description,category
0,The payment was deducted from my bank account ...,Account Suspension
1,I found a bug in the latest update affecting r...,Performance Issue
2,The application crashes whenever I try to uplo...,Performance Issue
3,My subscription was cancelled without my reque...,Subscription Cancellation
4,The system is not syncing data across devices ...,Feature Request
